要运行此程序，请在**免费** Tesla T4 Google Colab 实例上按“*运行时*”并按“*全部运行*”！
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> 如果您需要帮助，请加入 Discord + ⭐ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</a> </i> ⭐
</div>

要在本地设备上安装 Unsloth，请按照 [our guide](https://unsloth.ai/docs/get-started/install) 操作。此笔记本已获得许可 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)。

您将学习如何执行 [data prep](#Data)、如何执行 [train](#Train)、如何执行 [run the model](#Inference) 以及如何保存它

### 消息

隆重推出 **Unsloth Studio** - 一个新的开源、无代码 Web UI，用于训练和运行法学硕士。 [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<表><tr>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~% 2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fupload s%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia% 26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&宽度=376&dpr=3&质量=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>训练模型</b> - 无需代码</sub></td>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Z q%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26toke n%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&宽度=376&dpr=3&质量=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio 聊天 UI"></a><br><sub><b>在 Mac、Windows 和 Linux 上运行 GGUF 模型</b></sub></td>
</tr></表>

训练 MoE - DeepSeek、GLM、Qwen 和 gpt-oss 速度提高 12 倍，VRAM 减少 35%。 [Blog](https://unsloth.ai/docs/new/faster-moe)

超长上下文强化学习现已推出，上下文窗口增加了 7 倍！ [Blog](https://unsloth.ai/docs/new/grpo-long-context)

强化学习新功能：[FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

请访问我们的文档以获取所有 [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) 和 [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks)。

### 安装

In [ ]:
%%capture
import os

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### 不懒惰

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # 选择任何一个！我们内部自动支持 RoPE Scaling！
dtype = None # 没有用于自动检测。 Float16 适用于 Tesla T4、V100，Bfloat16 适用于 Ampere+
load_in_4bit = True # 使用 4 位量化来减少内存使用。可能是假的。

# 我们支持 4 位预量化模型，下载速度提高 4 倍 + 无 OOM。
fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/llama-2-13b-bnb-4bit",
    "unsloth/codellama-34b-bnb-4bit",
    "unsloth/tinyllama-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit", # 新的谷歌 6 万亿代币模型速度提高了 2.5 倍！
    "unsloth/gemma-2b-bnb-4bit",
] # 更多模型请访问 https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/tinyllama-bnb-4bit", # 16 位加载的“unsloth/tinyllama”
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # 门控模型的 HF 令牌
)

/usr/local/lib/python3.10/dist-packages/unsloth/__init__.py:67: UserWarning: CUDA is not linked properly.
We shall run `ldconfig /usr/lib64-nvidia` to try to fix it.
  warnings.warn(

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

==((====))==  Unsloth: Fast Llama patching release 2024.1
   \\   /|    GPU: Tesla T4. Max memory: 14.748 GB
O^O/ \_/ \    CUDA capability = 7.5. Xformers = 0.0.22.post7. FA = False.
\        /    Pytorch version: 2.1.0+cu121. CUDA Toolkit = 12.1
 "-____-"     bfloat16 = FALSE. Platform = Linux

Unsloth: unsloth/tinyllama-bnb-4bit can only handle sequence lengths of at most 2048.
But with kaiokendev's RoPE scaling of 2.0, it can be magically be extended to 4096!
You passed `quantization_config` to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` attribute will be overwritten with the one you passed to `from_pretrained`.

model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

**[注意]** TinyLlama 的内部最大序列长度为 2048。我们使用 RoPE Scaling 将其通过 Unsloth 扩展至 4096！

我们现在添加 LoRA 适配器，因此我们只需要更新所有参数的 1 到 10%！

**[注意]** 我们仅为 TinyLlama 设置 `gradient_checkpointing=False`，因为 Unsloth 可以节省大量内存使用量。这不适用于“llama-2-7b”或“mistral-7b”，因为内存使用量仍将超过 Tesla T4 的 15GB。 GC 在向后传递期间重新计算向前传递，从而节省内存负载。

`**[如果内存不足]**` 将 `gradient_checkpointing` 设置为 `True`。

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # 选择任意大于 0 的数字！建议 8、16、32、64、128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # 目前仅支持dropout = 0
    bias = "none",    # 目前仅支持bias =“none”
    use_gradient_checkpointing = False, # @@@ 如果你忘记了 - 设置为 True @@@
    random_state = 3407,
    use_rslora = False,  # 我们支持排名稳定的 LoRA
    loftq_config = None, # 和阁楼Q
)

Unsloth 2024.1 patched 22 layers with 22 QKV layers, 22 O layers and 22 MLP layers.


<a名称=“数据”></a>
### 数据准备
我们现在使用 [Alpaca dataset](https://huggingface.co/datasets/unsloth/alpaca-cleaned)，它是原始 [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html) 的 52K 的过滤版本。您可以将此代码部分替换为您自己的数据准备。

**[注意]** 要仅在完成情况下进行训练（忽略用户的输入），请阅读我们的文档 [here](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide#training-on-completions-only-masking-out-inputs)

**[注意]** 请记住将 **EOS_TOKEN** 添加到标记化输出！否则你将得到无限的后代！

如果您想对 ShareGPT 数据集使用“llama-3”模板，请尝试我们的会话 [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_v0.3_(7B)-Conversational.ipynb）

对于小说写作等文本补全，请尝试此 [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb)。

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

# ＃＃ 操作说明：
{}

# ＃＃ 输入：
{}

# ＃＃ 回复：
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # 必须添加EOS_TOKEN，否则你的一代将永远延续下去！
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

from datasets import load_dataset
dataset = load_dataset("unsloth/alpaca-cleaned", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

<a name="火车"></a>
### 训练模型
现在让我们训练我们的模型。我们做了 1 个完整的 epoch，这使得 Alpaca 在 80 多分钟内跑完！

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = True, # 将短序列打包在一起以节省时间！
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 1,
        learning_rate = 2e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # 使用 TrackIO/WandB 等
    ),
)

PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
loading file tokenizer.model from cache at /root/.cache/huggingface/hub/models--unsloth--tinyllama-bnb-4bit/snapshots/fc56510003ea9d49362400b8a362345150802c31/tokenizer.model
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--unsloth--tinyllama-bnb-4bit/snapshots/fc56510003ea9d49362400b8a362345150802c31/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at /root/.cache/huggingface/hub/models--unsloth--tinyllama-bnb-4bit/snapshots/fc56510003ea9d49362400b8a362345150802c31/special_tokens_map.json
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--unsloth--tinyllama-bnb-4bit/s

Generating train split: 0 examples [00:00, ? examples/s]

Using auto half precision backend


In [ ]:
# @title 显示当前内存统计信息
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.748 GB.
0.879 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

***** Running training *****
  Num examples = 3,000
  Num Epochs = 1
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 4
  Total optimization steps = 375
  Number of trainable parameters = 25,231,360
You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
1,2.280100
2,2.243700
3,2.245500
4,2.243600
5,2.302000
6,2.287400
7,2.283900
8,2.144600
9,2.237200
10,2.245700




Training completed. Do not forget to share your model on huggingface.co/models =)




In [ ]:
# @title 显示最终内存和时间统计数据
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

5034.6413 seconds used for training.
83.91 minutes used for training.
Peak reserved memory = 13.508 GB.
Peak reserved memory for training = 12.629 GB.
Peak reserved memory % of max memory = 91.592 %.
Peak reserved memory for training % of max memory = 85.632 %.


<a name="推理"></a>
### 推论
让我们运行模型吧！您可以更改指令和输入 - 将输出留空！

In [ ]:
# alpaca_prompt = 从上面复制
FastLanguageModel.for_inference(model) # 使本机推理速度提高 2 倍
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonacci sequence.", # 操作说明
        "1, 1, 2, 3, 5, 8", # 输入
        "", # 输出 - 将此项留空以进行生成！
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

['<s> Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nContinue the fibonacci sequence.\n\n### Input:\n1, 1, 2, 3, 5, 8\n\n### Response:\nThe fibonacci sequence is a sequence of numbers that can be generated by adding the previous two numbers and then subtracting the previous number from the previous number. The first number in the sequence is 1, and the second number is 1. The third number in the sequence is 1 + 1 = 2, the fourth number is 1 + 2 = 3, and so on. The sequence can be continued by adding 1 + 1 = 2, 1 + 2 = 3, and so on.</s>']

您还可以使用“TextStreamer”进行连续推理 - 这样您就可以逐个令牌查看生成令牌，而不是一直等待！

In [ ]:
# alpaca_prompt = 从上面复制
FastLanguageModel.for_inference(model) # 使本机推理速度提高 2 倍
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonacci sequence.", # 操作说明
        "1, 1, 2, 3, 5, 8", # 输入
        "", # 输出 - 将此项留空以进行生成！
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<a name="保存"></a>
### 保存、加载微调模型
要将最终模型保存为 LoRA 适配器，请使用 Hugging Face 的 `push_to_hub` 进行在线保存，或使用 `save_pretrained` 进行本地保存。

**[注意]** 这仅保存 LoRA 适配器，而不是完整模型。要保存到 16 位或 GGUF，请向下滚动！

In [ ]:
model.save_pretrained("tinyllama_lora")  # 本地储蓄
tokenizer.save_pretrained("tinyllama_lora")
# model.push_to_hub("your_name/tinyllama_lora", token = "YOUR_HF_TOKEN") # 在线保存
# tokenizer.push_to_hub("your_name/tinyllama_lora", token = "YOUR_HF_TOKEN") # 在线保存

现在，如果您想加载我们刚刚保存用于推理的 LoRA 适配器，请将“False”设置为“True”：

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "tinyllama_lora", # 您用于训练的模型
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # 使本机推理速度提高 2 倍

# alpaca_prompt = 您必须从上面复制！

inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is a famous tall tower in Paris?", # 操作说明
        "", # 输入
        "", # 输出 - 将此项留空以进行生成！
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 64)

您还可以使用 Hugging Face 的“AutoPeftModelForCausalLM”。仅当您未安装“unsloth”时才使用此选项。它可能会慢得令人绝望，因为不支持“4bit”模型下载，而且 Unsloth 的**推理速度快了 2 倍**。

In [ ]:
if False:
    # 我强烈不建议 - 如果可能的话使用 Unsloth
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "tinyllama_lora", # 您用于训练的模型
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("tinyllama_lora")

### 保存为 VLLM 的 float16

我们还支持直接保存到`float16`。对于 float16 选择“merged_16bit”，对于 int4 选择“merged_4bit”。我们还允许“lora”适配器作为后备。使用 `push_to_hub_merged` 上传到您的 Hugging Face 帐户！您可以前往 https://huggingface.co/settings/tokens 获取您的个人代币。有关更多部署选项，请参阅 [our docs](https://unsloth.ai/docs/basics/inference-and-deployment)。

In [ ]:
# 合并到16位
if False: model.save_pretrained_merged("tinyllama_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/tinyllama_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# 合并到4bit
if False: model.save_pretrained_merged("tinyllama_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/tinyllama_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# 只需 LoRA 适配器
if False:
    model.save_pretrained("tinyllama_lora")
    tokenizer.save_pretrained("tinyllama_lora")
if False:
    model.push_to_hub("HF_USERNAME/tinyllama_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/tinyllama_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp 转换
要保存到 `GGUF` / `llama.cpp`，我们现在原生支持它！我们克隆 `llama.cpp` 并默认将其保存到 `q8_0`。我们允许像“q4_k_m”这样的所有方法。使用`save_pretrained_gguf`进行本地保存，使用`push_to_hub_gguf`上传到HF。

一些支持的量化方法（完整列表在我们的 [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf) 上）：
* `q8_0` - 快速转换。资源使用率较高，但总体上可以接受。
* `q4_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q4_K。
* `q5_k_m` - 推荐。使用Q6_K作为attention.wv和feed_forward.w2张量的一半，否则使用Q5_K。

In [ ]:
# 保存到8位Q8_0
if False: model.save_pretrained_gguf("tinyllama_finetune", tokenizer,)
if False: model.push_to_hub_gguf("HF_USERNAME/tinyllama_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# 保存到16位GGUF
if False: model.save_pretrained_gguf("tinyllama_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/tinyllama_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# 保存到 q4_k_m GGUF
if False: model.save_pretrained_gguf("tinyllama_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/tinyllama_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

我们就完成了！如果您对 Unsloth 有任何疑问，我们有 [Discord](https://discord.gg/unsloth) 频道！如果您发现任何错误或想要随时更新最新的 LLM 内容，或需要帮助、加入项目等，请随时加入我们的 Discord！

其他一些资源：
1. 希望在本地使用 Unsloth？请阅读我们的 [Installation Guide](https://unsloth.ai/docs/get-started/install)，了解有关在 Windows、Docker、AMD、Intel GPU 上安装 Unsloth 的详细信息。
2. 了解如何使用我们的 [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) 进行强化学习。
3. 阅读我们的指南和笔记本以了解 [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) 和 [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) 模型支持。
4. 浏览我们的 [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) 以查找每个型号的专用指南。
5. 需要推理方面的帮助吗？请阅读我们的 [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment)，了解有关使用 vLLM、llama.cpp、Ollama 等的详细信息。

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  如果您需要帮助，请加入 Discord + ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</i> ⭐️

  <b>此笔记本和所有 Unsloth 笔记本均已获得 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme) 许可</b>
</div>